# 05. Tableau 데이터 준비

본 노트북은 EDA, 통계분석, 머신러닝 결과를 바탕으로 **Tableau 대시보드 제작용 CSV**를 생성한다.

목적은 다음과 같다.

1. 시설 단위 최종 데이터셋 생성
2. 지역명 표준화
3. 관람객 수 구간 및 성과 그룹 생성
4. 머신러닝 예측 결과 결합
5. 지역/시설유형/활성화 요인 요약 테이블 생성
6. Tableau 연결용 CSV 저장

> 앞선 분석 노트북과 달리, 이 노트북의 목적은 새로운 검정보다는 **시각화에 적합한 데이터 구조로 정리하는 것**이다.

## 1. 라이브러리 및 경로 설정

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# 현재 노트북을 프로젝트 루트에서 실행한다고 가정한다.
# 단, 파일을 찾지 못할 경우 /mnt/data도 함께 탐색하도록 구성한다.
PROJECT_ROOT = Path.cwd()
FALLBACK_ROOT = Path("/mnt/data")

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tableau"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT: c:\Users\hi\cultural_contest
OUTPUT_DIR: c:\Users\hi\cultural_contest\outputs\tableau


## 2. 데이터 불러오기

필요한 파일은 다음과 같다.

- `data/processed/facility_eda_preprocessed.csv`
- `outputs/ml/model_performance.csv`
- `outputs/ml/best_model_predictions.csv`
- `outputs/ml/best_model_permutation_importance.csv`
- `outputs/ml/residual_by_facility_type.csv`
- `outputs/ml/residual_by_region.csv`
- `outputs/statistics/correlation_results.csv`
- `outputs/statistics/service_test_results.csv`

경로가 다를 경우 아래 `read_csv_flexible()` 함수가 `/mnt/data`에서도 파일을 찾아 읽는다.

In [2]:
def read_csv_flexible(relative_path, fallback_name=None):
    # 프로젝트 상대 경로를 우선 탐색하고, 없으면 /mnt/data에서 파일명을 기준으로 탐색한다.
    relative_path = Path(relative_path)
    candidates = [PROJECT_ROOT / relative_path]

    if fallback_name is None:
        fallback_name = relative_path.name
    candidates.append(FALLBACK_ROOT / fallback_name)

    for path in candidates:
        if path.exists():
            print(f"loaded: {path}")
            return pd.read_csv(path)

    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {relative_path} 또는 {fallback_name}")

facility = read_csv_flexible("data/processed/facility_eda_preprocessed.csv")
model_performance = read_csv_flexible("outputs/ml/model_performance.csv")
best_predictions = read_csv_flexible("outputs/ml/best_model_predictions.csv")
importance = read_csv_flexible("outputs/ml/best_model_permutation_importance.csv")
residual_type = read_csv_flexible("outputs/ml/residual_by_facility_type.csv")
residual_region = read_csv_flexible("outputs/ml/residual_by_region.csv")
correlation = read_csv_flexible("outputs/statistics/correlation_results.csv")
service_test = read_csv_flexible("outputs/statistics/service_test_results.csv")

print()
print("facility:", facility.shape)
print("model_performance:", model_performance.shape)
print("best_predictions:", best_predictions.shape)
print("importance:", importance.shape)

loaded: c:\Users\hi\cultural_contest\data\processed\facility_eda_preprocessed.csv
loaded: c:\Users\hi\cultural_contest\outputs\ml\model_performance.csv
loaded: c:\Users\hi\cultural_contest\outputs\ml\best_model_predictions.csv
loaded: c:\Users\hi\cultural_contest\outputs\ml\best_model_permutation_importance.csv
loaded: c:\Users\hi\cultural_contest\outputs\ml\residual_by_facility_type.csv
loaded: c:\Users\hi\cultural_contest\outputs\ml\residual_by_region.csv
loaded: c:\Users\hi\cultural_contest\outputs\statistics\correlation_results.csv
loaded: c:\Users\hi\cultural_contest\outputs\statistics\service_test_results.csv

facility: (1191, 64)
model_performance: (21, 9)
best_predictions: (228, 13)
importance: (19, 3)


## 3. 지역명 표준화

EDA 및 통계분석 결과에서 다음 지역명이 분리되어 나타날 수 있었다.

- `강원` / `강원특별자치도`
- `전북` / `전북특별자치도`
- `제주` / `제주특별자치도`

Tableau 지도와 지역별 집계를 안정적으로 만들기 위해 표준 지역명을 새 컬럼으로 생성한다.

In [3]:
province_map = {
    "강원": "강원특별자치도",
    "전북": "전북특별자치도",
    "제주": "제주특별자치도",
}

region_group_map = {
    "서울": "수도권",
    "경기": "수도권",
    "인천": "수도권",
    "강원특별자치도": "강원권",
    "대전": "충청권",
    "세종": "충청권",
    "충북": "충청권",
    "충남": "충청권",
    "광주": "전라권",
    "전남": "전라권",
    "전북특별자치도": "전라권",
    "부산": "경상권",
    "대구": "경상권",
    "울산": "경상권",
    "경북": "경상권",
    "경남": "경상권",
    "제주특별자치도": "제주권",
}

facility = facility.copy()
facility["CTPRVN_NM_STD"] = facility["CTPRVN_NM"].replace(province_map)
facility["REGION_GROUP_STD"] = facility["CTPRVN_NM_STD"].map(region_group_map).fillna(facility["REGION_GROUP"])
facility["SIDO_SIGUNGU_STD"] = facility["CTPRVN_NM_STD"].astype(str) + " " + facility["SIGNGU_NM"].astype(str)

# 표준화 전후 확인
region_check = (
    facility
    .groupby(["CTPRVN_NM", "CTPRVN_NM_STD", "REGION_GROUP_STD"], dropna=False)
    .size()
    .reset_index(name="facility_count")
    .sort_values(["CTPRVN_NM_STD", "CTPRVN_NM"])
)

display(region_check)

,CTPRVN_NM,CTPRVN_NM_STD,REGION_GROUP_STD,facility_count
0,강원,강원특별자치도,강원권,19
1,강원특별자치도,강원특별자치도,강원권,99
2,경기,경기,수도권,186
3,경남,경남,경상권,85
4,경북,경북,경상권,82
5,광주,광주,전라권,27
6,대구,대구,경상권,25
7,대전,대전,충청권,20
8,부산,부산,경상권,42
9,서울,서울,수도권,178


## 4. 관람객 수 구간 및 성과 그룹 생성

Tableau 필터와 색상 구분에 사용할 변수를 만든다.

- `viewer_count_band`: 관람객 수 절대 구간
- `viewer_quartile`: 관람객 수 분위 그룹
- `activation_level`: 관람객 수 기준 활성화 수준

In [4]:
facility["viewer_count_band"] = pd.cut(
    facility["VIEWNG_NMPR_CO"],
    bins=[-1, 0, 10_000, 50_000, 100_000, 500_000, np.inf],
    labels=["0명", "1만 이하", "1만~5만", "5만~10만", "10만~50만", "50만 이상"],
)
facility["viewer_count_band"] = facility["viewer_count_band"].astype("object").fillna("관람객 수 미상")

valid_viewer_mask = facility["VIEWNG_NMPR_CO"].notna()
facility["viewer_quartile"] = "관람객 수 미상"
facility.loc[valid_viewer_mask, "viewer_quartile"] = pd.qcut(
    facility.loc[valid_viewer_mask, "VIEWNG_NMPR_CO"],
    q=4,
    labels=["Q1 낮음", "Q2 중하", "Q3 중상", "Q4 높음"],
    duplicates="drop",
).astype(str)

q33 = facility["VIEWNG_NMPR_CO"].quantile(1/3)
q67 = facility["VIEWNG_NMPR_CO"].quantile(2/3)

conditions = [
    facility["VIEWNG_NMPR_CO"].isna(),
    facility["VIEWNG_NMPR_CO"] <= q33,
    facility["VIEWNG_NMPR_CO"] <= q67,
    facility["VIEWNG_NMPR_CO"] > q67,
]
choices = ["미상", "낮음", "보통", "높음"]
facility["activation_level"] = np.select(conditions, choices, default="미상")

print("관람객 수 33% 분위:", round(q33, 2))
print("관람객 수 67% 분위:", round(q67, 2))
display(facility[["VIEWNG_NMPR_CO", "viewer_count_band", "viewer_quartile", "activation_level"]].head())

관람객 수 33% 분위: 8000.0
관람객 수 67% 분위: 41206.0


,VIEWNG_NMPR_CO,viewer_count_band,viewer_quartile,activation_level
0,"4,180,285.000",50만 이상,Q4 높음,높음
1,"1,307,690.000",50만 이상,Q4 높음,높음
2,"83,306.000",5만~10만,Q4 높음,높음
3,"788,347.000",50만 이상,Q4 높음,높음
4,"397,107.000",10만~50만,Q4 높음,높음


## 5. 머신러닝 예측 결과 결합

최적 모델의 예측 결과는 test set에 대해서만 존재한다. 따라서 전체 시설 데이터에 left join으로 붙이고, 예측 결과가 있는 시설과 없는 시설을 구분하는 플래그를 만든다.

예측값이 있는 시설은 다음과 같은 시각화에 활용할 수 있다.

- 실제 관람객 수 vs 예측 관람객 수
- 예상보다 성과가 높은 시설
- 예상보다 성과가 낮은 시설
- 시설 유형/지역별 예측 오차

In [5]:
pred_cols = [
    "FCLTY_NM", "facility_type", "CTPRVN_NM", "VIEWNG_NMPR_CO",
    "pred_log", "actual_viewers", "pred_viewers",
    "error_log", "abs_error_log", "error_viewers", "abs_error_viewers"
]

merge_keys = ["FCLTY_NM", "facility_type", "CTPRVN_NM", "VIEWNG_NMPR_CO"]

# 중복 키 확인
print("facility duplicated merge keys:", facility.duplicated(merge_keys, keep=False).sum())
print("prediction duplicated merge keys:", best_predictions.duplicated(merge_keys, keep=False).sum())

facility_main = facility.merge(
    best_predictions[pred_cols],
    on=merge_keys,
    how="left"
)

facility_main["has_prediction"] = facility_main["pred_log"].notna()

# 예측 성과 구분: 실제 - 예측 기준
facility_main["prediction_performance_group"] = "예측 대상 아님"

pred_mask = facility_main["has_prediction"]
if pred_mask.sum() > 0:
    err_q25 = facility_main.loc[pred_mask, "error_viewers"].quantile(0.25)
    err_q75 = facility_main.loc[pred_mask, "error_viewers"].quantile(0.75)

    facility_main.loc[pred_mask & (facility_main["error_viewers"] >= err_q75), "prediction_performance_group"] = "예상보다 높음"
    facility_main.loc[pred_mask & (facility_main["error_viewers"] <= err_q25), "prediction_performance_group"] = "예상보다 낮음"
    facility_main.loc[pred_mask & (facility_main["error_viewers"].between(err_q25, err_q75)), "prediction_performance_group"] = "예상 범위"

    print("예측 오차 25% 분위:", round(err_q25, 2))
    print("예측 오차 75% 분위:", round(err_q75, 2))

print("예측 결과가 붙은 시설 수:", facility_main["has_prediction"].sum())
display(facility_main[["FCLTY_NM", "facility_type", "CTPRVN_NM", "VIEWNG_NMPR_CO", "pred_viewers", "error_viewers", "prediction_performance_group"]].dropna(subset=["pred_viewers"]).head())

facility duplicated merge keys: 0
prediction duplicated merge keys: 0
예측 오차 25% 분위: -8269.74
예측 오차 75% 분위: 21707.38
예측 결과가 붙은 시설 수: 228


,FCLTY_NM,facility_type,CTPRVN_NM,VIEWNG_NMPR_CO,pred_viewers,error_viewers,prediction_performance_group
4,국립한글박물관,박물관,서울,"397,107.000","1,429,514.531","-1,032,407.531",예상보다 낮음
16,국립대한민국임시정부기념관,박물관,서울,"164,627.000","156,920.647","7,706.353",예상 범위
17,둘리뮤지엄,박물관,서울,"74,520.000","36,502.587","38,017.413",예상보다 높음
19,서대문형무소역사관,박물관,서울,"650,000.000","83,624.269","566,375.731",예상보다 높음
21,서울공예박물관,박물관,서울,"532,717.000","355,472.371","177,244.629",예상보다 높음


## 6. Tableau 시설 단위 메인 데이터셋 생성

시설 단위 데이터셋은 Tableau의 가장 기본 데이터 원본으로 사용한다.

추천 활용:

- 지도 시각화
- 시설 유형 필터
- 지역/권역 필터
- 관람객 수 구간 필터
- 개별 시설 상세 정보 확인
- 실제값/예측값/오차 비교

In [6]:
facility_main_cols = [
    # 식별/분류
    "ID", "FCLTY_NM", "facility_type",
    "CTPRVN_NM", "CTPRVN_NM_STD", "SIGNGU_NM", "SIDO_SIGUNGU", "SIDO_SIGUNGU_STD",
    "REGION_GROUP", "REGION_GROUP_STD",

    # 위치
    "FCLTY_LO", "FCLTY_LA",

    # 운영/규모
    "OPNNG_YEAR", "BASE_YEAR", "OPERATING_YEARS", "OPNNG_DAY_CO",
    "LND_AR_VALUE", "EDC_FCLTY_AR_VALUE", "DATA_SPCE_AR_VALUE",
    "NMPR_CO", "DATA_CO", "TOT_PROGRM_CO",
    "ARTGR_EMP_CO", "QUALF_HOLD_CO", "GNRL_GVRNM_EMP_CO",

    # 로그 변환 변수
    "LOG1P_LND_AR_VALUE", "LOG1P_EDC_FCLTY_AR_VALUE", "LOG1P_DATA_SPCE_AR_VALUE",
    "LOG1P_NMPR_CO", "LOG1P_DATA_CO", "LOG1P_TOT_PROGRM_CO",

    # 서비스
    "MOBILE_PROVD_AT", "SOUND_PROVD_AT",

    # 관람객 수
    "VIEWNG_NMPR_CO", "DAY_AVRG_VIEWNG_NMPR_CO",
    "LOG1P_VIEWNG_NMPR_CO", "LOG1P_DAY_AVRG_VIEWNG_NMPR_CO",
    "viewer_count_band", "viewer_quartile", "activation_level",

    # 예측 결과
    "has_prediction", "pred_log", "actual_viewers", "pred_viewers",
    "error_log", "abs_error_log", "error_viewers", "abs_error_viewers",
    "prediction_performance_group",

    # 참고 플래그
    "IQR_OUTLIER_FLAG_COUNT",
]

facility_main_cols = [c for c in facility_main_cols if c in facility_main.columns]
tableau_facility_main = facility_main[facility_main_cols].copy()

display(tableau_facility_main.head())
print("tableau_facility_main:", tableau_facility_main.shape)

,ID,FCLTY_NM,facility_type,CTPRVN_NM,CTPRVN_NM_STD,SIGNGU_NM,SIDO_SIGUNGU,SIDO_SIGUNGU_STD,REGION_GROUP,REGION_GROUP_STD,FCLTY_LO,FCLTY_LA,OPNNG_YEAR,BASE_YEAR,OPERATING_YEARS,OPNNG_DAY_CO,LND_AR_VALUE,EDC_FCLTY_AR_VALUE,DATA_SPCE_AR_VALUE,NMPR_CO,DATA_CO,TOT_PROGRM_CO,ARTGR_EMP_CO,QUALF_HOLD_CO,GNRL_GVRNM_EMP_CO,LOG1P_LND_AR_VALUE,LOG1P_EDC_FCLTY_AR_VALUE,LOG1P_DATA_SPCE_AR_VALUE,LOG1P_NMPR_CO,LOG1P_DATA_CO,LOG1P_TOT_PROGRM_CO,MOBILE_PROVD_AT,SOUND_PROVD_AT,VIEWNG_NMPR_CO,DAY_AVRG_VIEWNG_NMPR_CO,LOG1P_VIEWNG_NMPR_CO,LOG1P_DAY_AVRG_VIEWNG_NMPR_CO,viewer_count_band,viewer_quartile,activation_level,has_prediction,pred_log,actual_viewers,pred_viewers,error_log,abs_error_log,error_viewers,abs_error_viewers,prediction_performance_group,IQR_OUTLIER_FLAG_COUNT
0,KCDMMUS23N000000001,국립중앙박물관,박물관,서울,서울,용산구,서울 용산구,서울 용산구,수도권,수도권,126.978,37.525,1945,2025,80,360.000,"285,991.000","4,975.000","1,552.000",89.000,"157,323.000",16.000,NaN,5.000,128.000,12.564,8.512,7.348,4.500,11.966,2.833,O,O,"4,180,285.000","11,612.000",15.246,9.360,50만 이상,Q4 높음,높음,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,예측 대상 아님,10
1,KCDMMUS23N000000002,국립민속박물관,박물관,서울,서울,종로구,서울 종로구,서울 종로구,수도권,수도권,126.979,37.582,1946,2025,79,362.000,"39,626.620","1,376.680",324.000,58.000,"92,786.000",70.000,NaN,1.000,37.000,10.587,7.228,5.784,4.078,11.438,4.263,O,X,"1,307,690.000","3,612.000",14.084,8.192,50만 이상,Q4 높음,높음,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,예측 대상 아님,8
2,KCDMMUS23N000000003,국립민속박물관 파주\n(개방형 수장고 및 정보센터),박물관,경기,경기,파주시,경기 파주시,경기 파주시,수도권,수도권,126.694,37.787,2021,2025,4,313.000,"60,212.700",277.490,NaN,NaN,NaN,10.000,NaN,NaN,NaN,11.006,5.629,NaN,NaN,NaN,2.398,X,X,"83,306.000",266.000,11.330,5.587,5만~10만,Q4 높음,높음,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,예측 대상 아님,1
3,KCDMMUS23N000000004,대한민국역사박물관,박물관,서울,서울,종로구,서울 종로구,서울 종로구,수도권,수도권,126.978,37.574,2012,2025,13,362.000,"6,445.000",496.000,81.000,55.000,"18,065.000",34.000,NaN,23.000,NaN,8.771,6.209,4.407,4.025,9.802,3.555,O,O,"788,347.000","2,178.000",13.578,7.687,50만 이상,Q4 높음,높음,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,예측 대상 아님,5
4,KCDMMUS23N000000005,국립한글박물관,박물관,서울,서울,용산구,서울 용산구,서울 용산구,수도권,수도권,126.980,37.521,2014,2025,11,362.000,"285,991.000",378.000,220.000,33.000,"30,059.000",33.000,NaN,NaN,19.000,12.564,5.938,5.398,3.526,10.311,3.526,O,O,"397,107.000","1,097.000",12.892,7.001,10만~50만,Q4 높음,높음,True,14.173,"397,107.000","1,429,514.531",-1.281,1.281,"-1,032,407.531","1,032,407.531",예상보다 낮음,9


tableau_facility_main: (1191, 50)


## 7. 지역 단위 요약 데이터셋 생성

지역별 시설 수, 관람객 수, 서비스 제공률, 예측 오차를 요약한다.

추천 활용:

- 지역별 시설 수 지도
- 지역별 관람객 수 중앙값 막대그래프
- 권역별 비교
- 예상 대비 성과가 높은 지역 확인

In [7]:
def ratio_o(series):
    valid = series.dropna()
    if len(valid) == 0:
        return np.nan
    return (valid == "O").mean()

region_summary = (
    facility_main
    .groupby(["REGION_GROUP_STD", "CTPRVN_NM_STD"], dropna=False)
    .agg(
        facility_count=("ID", "count"),
        museum_count=("facility_type", lambda s: (s == "박물관").sum()),
        art_gallery_count=("facility_type", lambda s: (s == "미술관").sum()),
        mean_viewers=("VIEWNG_NMPR_CO", "mean"),
        median_viewers=("VIEWNG_NMPR_CO", "median"),
        total_viewers=("VIEWNG_NMPR_CO", "sum"),
        mean_log_viewers=("LOG1P_VIEWNG_NMPR_CO", "mean"),
        median_log_viewers=("LOG1P_VIEWNG_NMPR_CO", "median"),
        mean_operating_years=("OPERATING_YEARS", "mean"),
        median_land_area=("LND_AR_VALUE", "median"),
        median_education_area=("EDC_FCLTY_AR_VALUE", "median"),
        median_program_count=("TOT_PROGRM_CO", "median"),
        mobile_provided_rate=("MOBILE_PROVD_AT", ratio_o),
        sound_provided_rate=("SOUND_PROVD_AT", ratio_o),
        prediction_count=("has_prediction", "sum"),
        mean_error_viewers=("error_viewers", "mean"),
        median_abs_error_viewers=("abs_error_viewers", "median"),
    )
    .reset_index()
    .sort_values(["median_viewers", "facility_count"], ascending=[False, False])
)

region_summary["museum_ratio"] = region_summary["museum_count"] / region_summary["facility_count"]
region_summary["art_gallery_ratio"] = region_summary["art_gallery_count"] / region_summary["facility_count"]

display(region_summary)

,REGION_GROUP_STD,CTPRVN_NM_STD,facility_count,museum_count,art_gallery_count,mean_viewers,median_viewers,total_viewers,mean_log_viewers,median_log_viewers,mean_operating_years,median_land_area,median_education_area,median_program_count,mobile_provided_rate,sound_provided_rate,prediction_count,mean_error_viewers,median_abs_error_viewers,museum_ratio,art_gallery_ratio
12,제주권,제주특별자치도,82,61,21,"123,321.532","64,071.000","9,495,758.000",10.925,11.068,18.524,"18,000.000",150.000,3.500,0.198,0.203,16,"-19,898.508","19,359.189",0.744,0.256
5,경상권,울산,11,10,1,"72,789.636","34,640.000","800,686.000",10.766,10.453,13.727,"6,182.000",80.000,8.500,0.182,0.000,1,"12,642.638","12,642.638",0.909,0.091
8,수도권,인천,37,30,7,"57,082.528","29,320.500","2,054,971.000",9.754,10.282,17.351,"3,475.000",103.500,6.000,0.167,0.108,9,"41,629.654","17,732.492",0.811,0.189
2,경상권,경북,82,70,12,"66,986.215","25,106.000","5,291,911.000",9.869,10.131,20.378,"9,804.000",180.000,4.000,0.183,0.122,16,"41,688.249","24,074.542",0.854,0.146
1,경상권,경남,85,75,10,"56,053.902","24,434.000","4,596,420.000",10.000,10.102,20.129,"9,808.000",162.000,6.000,0.202,0.119,14,"8,056.417","7,013.552",0.882,0.118
6,수도권,경기,186,127,59,"53,165.505","20,115.000","9,782,453.000",9.608,9.909,17.817,"4,942.000",133.000,7.000,0.249,0.124,39,"3,314.094","12,882.617",0.683,0.317
15,충청권,충남,75,64,11,"140,097.657","18,929.500","9,806,836.000",9.901,9.848,20.707,"8,149.500",122.000,4.000,0.213,0.093,13,"213,878.693","13,108.204",0.853,0.147
11,전라권,전북특별자치도,63,42,21,"67,035.034","17,919.000","3,888,032.000",9.880,9.794,17.349,"8,059.000",158.000,6.000,0.206,0.079,13,"66,186.534","28,278.965",0.667,0.333
4,경상권,부산,42,33,9,"64,031.286","17,530.000","2,241,095.000",9.702,9.772,23.095,"2,684.000",147.220,6.000,0.190,0.167,8,"-17,851.034","4,018.894",0.786,0.214
0,강원권,강원특별자치도,118,99,19,"54,634.802","16,579.000","6,064,463.000",9.711,9.716,17.898,"7,657.500",129.000,6.000,0.120,0.085,16,"28,664.908","13,361.903",0.839,0.161


## 8. 시설 유형 단위 요약 데이터셋 생성

박물관과 미술관의 운영·규모·관람객 수 차이를 Tableau에서 비교하기 위한 요약 테이블이다.

In [8]:
type_summary = (
    facility_main
    .groupby("facility_type", dropna=False)
    .agg(
        facility_count=("ID", "count"),
        mean_viewers=("VIEWNG_NMPR_CO", "mean"),
        median_viewers=("VIEWNG_NMPR_CO", "median"),
        total_viewers=("VIEWNG_NMPR_CO", "sum"),
        mean_log_viewers=("LOG1P_VIEWNG_NMPR_CO", "mean"),
        median_log_viewers=("LOG1P_VIEWNG_NMPR_CO", "median"),
        mean_operating_years=("OPERATING_YEARS", "mean"),
        median_open_days=("OPNNG_DAY_CO", "median"),
        median_land_area=("LND_AR_VALUE", "median"),
        median_education_area=("EDC_FCLTY_AR_VALUE", "median"),
        median_program_count=("TOT_PROGRM_CO", "median"),
        median_data_count=("DATA_CO", "median"),
        mobile_provided_rate=("MOBILE_PROVD_AT", ratio_o),
        sound_provided_rate=("SOUND_PROVD_AT", ratio_o),
        prediction_count=("has_prediction", "sum"),
        mean_error_viewers=("error_viewers", "mean"),
        median_abs_error_viewers=("abs_error_viewers", "median"),
    )
    .reset_index()
)

display(type_summary)

,facility_type,facility_count,mean_viewers,median_viewers,total_viewers,mean_log_viewers,median_log_viewers,mean_operating_years,median_open_days,median_land_area,median_education_area,median_program_count,median_data_count,mobile_provided_rate,sound_provided_rate,prediction_count,mean_error_viewers,median_abs_error_viewers
0,미술관,294,"60,374.826","13,400.000","16,965,326.000",9.572,9.503,16.340,300.000,"3,501.000",107.000,7.000,"1,500.000",0.218,0.170,56,"10,216.009","9,840.515"
1,박물관,897,"86,859.376","21,353.000","74,351,626.000",9.880,9.969,21.287,307.000,"6,601.500",156.000,6.000,"2,143.000",0.211,0.130,172,"35,400.629","14,546.957"


## 9. 활성화 요인 요약 데이터셋 생성

통계분석의 상관계수와 머신러닝의 permutation importance를 결합하여 Tableau에서 보여줄 수 있는 요인 요약 테이블을 만든다.

이 표는 다음 질문에 답하기 위한 보조 데이터다.

> 관람객 수와 관련된 주요 운영·시설 요인은 무엇인가?

In [9]:
factor_name_map = {
    "OPNNG_DAY_CO": "개관일수",
    "LOG1P_NMPR_CO": "인원 수(로그)",
    "LOG1P_LND_AR_VALUE": "토지면적(로그)",
    "ARTGR_EMP_CO": "학예 인력",
    "facility_type": "시설 유형",
    "LOG1P_EDC_FCLTY_AR_VALUE": "교육시설면적(로그)",
    "LOG1P_TOT_PROGRM_CO": "프로그램 수(로그)",
    "LOG1P_DATA_CO": "소장자료 수(로그)",
    "LOG1P_DATA_SPCE_AR_VALUE": "자료공간면적(로그)",
    "MOBILE_PROVD_AT": "모바일 제공 여부",
    "SOUND_PROVD_AT": "음성 제공 여부",
    "QUALF_HOLD_CO": "자격 보유자 수",
    "GNRL_GVRNM_EMP_CO": "일반 정부 인력",
}

factor_group_map = {
    "OPNNG_DAY_CO": "운영",
    "LOG1P_NMPR_CO": "인력",
    "LOG1P_LND_AR_VALUE": "시설 규모",
    "ARTGR_EMP_CO": "인력",
    "facility_type": "시설 유형",
    "LOG1P_EDC_FCLTY_AR_VALUE": "시설 규모",
    "LOG1P_TOT_PROGRM_CO": "프로그램",
    "LOG1P_DATA_CO": "소장자료",
    "LOG1P_DATA_SPCE_AR_VALUE": "시설 규모",
    "MOBILE_PROVD_AT": "서비스",
    "SOUND_PROVD_AT": "서비스",
    "QUALF_HOLD_CO": "인력",
    "GNRL_GVRNM_EMP_CO": "인력",
}

# 상관분석 변수명과 중요도 변수명이 다를 수 있어 원 변수 기준으로 일부 매핑한다.
corr_key_map = {
    "LOG1P_LND_AR_VALUE": "LOG1P_LND_AR_VALUE",
    "LOG1P_EDC_FCLTY_AR_VALUE": "LOG1P_EDC_FCLTY_AR_VALUE",
    "LOG1P_TOT_PROGRM_CO": "LOG1P_TOT_PROGRM_CO",
    "LOG1P_DATA_CO": "LOG1P_DATA_CO",
    "LOG1P_DATA_SPCE_AR_VALUE": "LOG1P_DATA_SPCE_AR_VALUE",
    "LOG1P_NMPR_CO": "LOG1P_NMPR_CO",
    "OPNNG_DAY_CO": "OPNNG_DAY_CO",
    "ARTGR_EMP_CO": "ARTGR_EMP_CO",
    "QUALF_HOLD_CO": "QUALF_HOLD_CO",
    "GNRL_GVRNM_EMP_CO": "GNRL_GVRNM_EMP_CO",
}

importance_tmp = importance.copy()
importance_tmp["display_name"] = importance_tmp["feature"].map(factor_name_map).fillna(importance_tmp["feature"])
importance_tmp["factor_group"] = importance_tmp["feature"].map(factor_group_map).fillna("기타")
importance_tmp["importance_rank"] = importance_tmp["importance_mean"].rank(ascending=False, method="min").astype(int)
importance_tmp["correlation_variable"] = importance_tmp["feature"].map(corr_key_map)

corr_cols = ["variable", "n", "missing_rate", "pearson_r", "pearson_p", "spearman_rho", "spearman_p"]
corr_tmp = correlation[corr_cols].copy()

factor_summary = importance_tmp.merge(
    corr_tmp,
    left_on="correlation_variable",
    right_on="variable",
    how="left"
)

factor_summary = factor_summary[
    [
        "importance_rank", "feature", "display_name", "factor_group",
        "importance_mean", "importance_std",
        "n", "missing_rate", "pearson_r", "pearson_p", "spearman_rho", "spearman_p"
    ]
].sort_values("importance_rank")

display(factor_summary)

,importance_rank,feature,display_name,factor_group,importance_mean,importance_std,n,missing_rate,pearson_r,pearson_p,spearman_rho,spearman_p
0,1,OPNNG_DAY_CO,개관일수,운영,0.180,0.016,"1,136.000",0.001,0.449,0.000,0.487,0.000
1,2,LOG1P_NMPR_CO,인원 수(로그),인력,0.114,0.030,633.000,0.443,0.182,0.000,0.140,0.000
2,3,LOG1P_LND_AR_VALUE,토지면적(로그),시설 규모,0.079,0.016,"1,127.000",0.009,0.448,0.000,0.475,0.000
3,4,ARTGR_EMP_CO,학예 인력,인력,0.058,0.010,520.000,0.543,-0.009,0.836,0.242,0.000
4,5,facility_type,시설 유형,시설 유형,0.038,0.008,NaN,NaN,NaN,NaN,NaN,NaN
5,6,LOG1P_TOT_PROGRM_CO,프로그램 수(로그),프로그램,0.036,0.010,895.000,0.213,0.273,0.000,0.244,0.000
6,7,MOBILE_PROVD_AT,모바일 제공 여부,서비스,0.029,0.011,NaN,NaN,NaN,NaN,NaN,NaN
7,8,LOG1P_EDC_FCLTY_AR_VALUE,교육시설면적(로그),시설 규모,0.019,0.005,834.000,0.266,0.368,0.000,0.361,0.000
8,9,PRVATE_VLNTER_CO,PRVATE_VLNTER_CO,기타,0.018,0.007,NaN,NaN,NaN,NaN,NaN,NaN
9,10,LOG1P_DATA_CO,소장자료 수(로그),소장자료,0.018,0.008,641.000,0.436,0.188,0.000,0.165,0.000


## 10. 모델 성능 비교 데이터셋 정리

`model_performance.csv`에 최적 모델 여부와 모델 그룹을 추가하여 Tableau에서 성능 비교 차트를 만들기 쉽게 정리한다.

In [10]:
performance = model_performance.copy()
performance = performance.sort_values(["RMSE_log", "MAE_log"]).reset_index(drop=True)
performance["rank_rmse"] = np.arange(1, len(performance) + 1)
performance["is_best_model"] = performance["rank_rmse"] == 1

linear_models = ["LinearRegression", "Ridge", "Lasso"]
tree_models = ["RandomForest", "ExtraTrees", "GradientBoosting"]
performance["model_group"] = np.select(
    [performance["model"].eq("Dummy_mean"), performance["model"].isin(linear_models), performance["model"].isin(tree_models)],
    ["기준 모델", "선형 계열", "트리 계열"],
    default="기타"
)

best_row = performance.iloc[0]
print("최적 모델:", best_row["feature_set"], best_row["model"])
print("RMSE_log:", round(best_row["RMSE_log"], 3))
print("R2_log:", round(best_row["R2_log"], 3))

display(performance.head(10))

최적 모델: staff Lasso
RMSE_log: 1.303
R2_log: 0.5


,feature_set,model,n_features_raw,MAE_log,RMSE_log,R2_log,MAE_original,RMSE_original,MedianAE_original,rank_rmse,is_best_model,model_group
0,staff,Lasso,19,1.053,1.303,0.500,"58,036.404","216,857.077","13,404.787",1,True,선형 계열
1,staff,GradientBoosting,19,1.030,1.304,0.499,"53,520.708","208,129.389","14,094.359",2,False,트리 계열
2,staff,Ridge,19,1.055,1.305,0.498,"58,056.666","217,002.441","13,358.675",3,False,선형 계열
3,staff,RandomForest,19,1.016,1.307,0.497,"52,937.209","208,619.791","13,055.190",4,False,트리 계열
4,staff,LinearRegression,19,1.057,1.307,0.496,"58,202.003","217,017.870","13,609.721",5,False,선형 계열
5,extended,ExtraTrees,12,1.067,1.322,0.485,"55,580.187","209,990.039","13,655.683",6,False,트리 계열
6,extended,Lasso,12,1.074,1.323,0.485,"57,756.005","212,247.021","15,406.942",7,False,선형 계열
7,extended,Ridge,12,1.075,1.324,0.484,"57,782.396","212,359.483","15,490.326",8,False,선형 계열
8,extended,LinearRegression,12,1.077,1.325,0.482,"57,870.581","212,399.925","15,593.731",9,False,선형 계열
9,extended,GradientBoosting,12,1.053,1.329,0.480,"54,718.119","208,344.495","14,419.948",10,False,트리 계열


## 11. 예측 성과 상위/하위 시설 데이터셋 생성

예측 결과가 있는 test set 시설 중에서 실제 관람객 수가 예측보다 높은 시설과 낮은 시설을 구분한다.

주의할 점은 이것이 인과적 의미의 우수/부진 시설을 뜻하는 것은 아니라는 점이다. 여기서는 **모델이 예측한 조건 대비 상대적으로 관람객 수가 높거나 낮은 시설**로 해석한다.

In [11]:
prediction_facilities = facility_main[facility_main["has_prediction"]].copy()

prediction_cols = [
    "FCLTY_NM", "facility_type", "CTPRVN_NM_STD", "SIGNGU_NM", "REGION_GROUP_STD",
    "VIEWNG_NMPR_CO", "pred_viewers", "error_viewers", "abs_error_viewers",
    "prediction_performance_group", "viewer_count_band", "activation_level",
    "FCLTY_LO", "FCLTY_LA"
]
prediction_cols = [c for c in prediction_cols if c in prediction_facilities.columns]

tableau_prediction_performance = prediction_facilities[prediction_cols].copy()
tableau_prediction_performance = tableau_prediction_performance.sort_values("error_viewers", ascending=False)

print("예상보다 높은 시설 상위 10개")
display(tableau_prediction_performance.head(10))

print("예상보다 낮은 시설 상위 10개")
display(tableau_prediction_performance.tail(10).sort_values("error_viewers"))

예상보다 높은 시설 상위 10개


,FCLTY_NM,facility_type,CTPRVN_NM_STD,SIGNGU_NM,REGION_GROUP_STD,VIEWNG_NMPR_CO,pred_viewers,error_viewers,abs_error_viewers,prediction_performance_group,viewer_count_band,activation_level,FCLTY_LO,FCLTY_LA
534,백제군사박물관,박물관,충남,논산시,충청권,"2,917,684.000","61,580.684","2,856,103.316","2,856,103.316",예상보다 높음,50만 이상,높음,127.182,36.192
19,서대문형무소역사관,박물관,서울,서대문구,수도권,"650,000.000","83,624.269","566,375.731","566,375.731",예상보다 높음,50만 이상,높음,126.956,37.575
914,리움미술관,미술관,서울,용산구,수도권,"391,414.000","50,447.419","340,966.581","340,966.581",예상보다 높음,10만~50만,높음,126.999,37.538
265,경기도어린이박물관,박물관,경기,용인시 기흥구,수도권,"373,719.000","77,033.718","296,685.282","296,685.282",예상보다 높음,10만~50만,높음,127.109,37.268
587,국립태권도박물관,박물관,전북특별자치도,무주군,전라권,"316,077.000","45,732.747","270,344.253","270,344.253",예상보다 높음,10만~50만,높음,127.779,36.009
954,대구문화예술회관 미술관,미술관,대구,달서구,경상권,"250,636.000","21,970.701","228,665.299","228,665.299",예상보다 높음,10만~50만,높음,128.558,35.845
641,목포근대역사관2관,박물관,전남,목포시,전라권,"226,015.000","5,885.184","220,129.816","220,129.816",예상보다 높음,10만~50만,높음,126.381,34.786
769,고성수석전시관,박물관,경남,고성군,경상권,"238,438.000","18,478.816","219,959.184","219,959.184",예상보다 높음,10만~50만,높음,128.373,35.071
584,국립익산박물관,박물관,전북특별자치도,익산시,전라권,"394,550.000","182,575.605","211,974.395","211,974.395",예상보다 높음,10만~50만,높음,127.029,36.012
693,국립등대박물관,박물관,경북,포항시 남구,경상권,"238,491.000","28,261.456","210,229.544","210,229.544",예상보다 높음,10만~50만,높음,129.568,36.078


예상보다 낮은 시설 상위 10개


,FCLTY_NM,facility_type,CTPRVN_NM_STD,SIGNGU_NM,REGION_GROUP_STD,VIEWNG_NMPR_CO,pred_viewers,error_viewers,abs_error_viewers,prediction_performance_group,viewer_count_band,activation_level,FCLTY_LO,FCLTY_LA
4,국립한글박물관,박물관,서울,용산구,수도권,"397,107.000","1,429,514.531","-1,032,407.531","1,032,407.531",예상보다 낮음,10만~50만,높음,126.980,37.521
26,서울역사박물관,박물관,서울,종로구,수도권,"581,669.000","810,971.591","-229,302.591","229,302.591",예상보다 낮음,50만 이상,높음,126.971,37.570
38,G밸리산업박물관,박물관,서울,구로구,수도권,"20,000.000","166,712.349","-146,712.349","146,712.349",예상보다 낮음,1만~5만,보통,126.895,37.480
138,부산광역시립박물관,박물관,부산,남구,경상권,"130,339.000","272,111.537","-141,772.537","141,772.537",예상보다 낮음,10만~50만,높음,129.093,35.130
991,김홍도미술관,미술관,경기,안산시,수도권,"58,251.000","185,731.983","-127,480.983","127,480.983",예상보다 낮음,5만~10만,높음,126.852,37.316
837,감귤박물관,박물관,제주특별자치도,서귀포시,제주권,"60,924.000","168,714.587","-107,790.587","107,790.587",예상보다 낮음,5만~10만,높음,126.607,33.272
846,제주세계자연유산센터 세계유산전시관,박물관,제주특별자치도,제주시,제주권,"33,335.000","132,910.127","-99,575.127","99,575.127",예상보다 낮음,1만~5만,보통,126.715,33.457
284,수원박물관,박물관,경기,수원시 영통구,수도권,"44,750.000","127,935.143","-83,185.143","83,185.143",예상보다 낮음,1만~5만,높음,127.036,37.298
546,유류피해극복기념관,박물관,충남,태안군,충청권,"77,048.000","151,128.082","-74,080.082","74,080.082",예상보다 낮음,5만~10만,높음,126.148,36.793
841,제주 삼양동 유적,박물관,제주특별자치도,제주시,제주권,"21,111.000","84,612.475","-63,501.475","63,501.475",예상보다 낮음,1만~5만,보통,126.588,33.524


## 12. Tableau 대시보드 설계용 메모 데이터셋

대시보드 시트 구성, 핵심 질문, 사용할 데이터셋을 표 형태로 정리한다.

In [12]:
dashboard_design_plan = pd.DataFrame([
    {
        "dashboard_page": "1. 전국 문화시설 현황",
        "key_question": "시설은 어디에 많고, 관람객 수는 어느 지역에서 높은가?",
        "main_dataset": "tableau_facility_main.csv / tableau_region_summary.csv",
        "recommended_charts": "지도, 지역별 막대그래프, KPI 카드",
        "filters": "시설 유형, 권역, 시도, 관람객 수 구간",
    },
    {
        "dashboard_page": "2. 시설 유형별 비교",
        "key_question": "박물관과 미술관은 운영·규모·관람객 수에서 어떤 차이가 있는가?",
        "main_dataset": "tableau_type_summary.csv / tableau_facility_main.csv",
        "recommended_charts": "박스플롯, 막대그래프, 요약 테이블",
        "filters": "시설 유형, 지역, 권역",
    },
    {
        "dashboard_page": "3. 활성화 요인 분석",
        "key_question": "관람객 수와 관련된 주요 운영·시설 요인은 무엇인가?",
        "main_dataset": "tableau_factor_summary.csv / tableau_facility_main.csv",
        "recommended_charts": "변수 중요도 막대그래프, 상관계수 막대그래프, 산점도",
        "filters": "요인 그룹, 시설 유형, 지역",
    },
    {
        "dashboard_page": "4. 서비스 제공 현황",
        "key_question": "모바일/음성 제공 시설은 어떤 특성을 보이는가?",
        "main_dataset": "tableau_facility_main.csv / tableau_region_summary.csv",
        "recommended_charts": "서비스 제공률 막대그래프, 관람객 수 비교 그래프",
        "filters": "서비스 제공 여부, 시설 유형, 지역",
    },
    {
        "dashboard_page": "5. 조건 대비 성과 탐색",
        "key_question": "비슷한 조건 대비 예상보다 관람객 수가 높은/낮은 시설은 어디인가?",
        "main_dataset": "tableau_prediction_performance.csv",
        "recommended_charts": "실제 vs 예측 산점도, 오차 막대그래프, 시설 리스트",
        "filters": "예측 성과 그룹, 시설 유형, 지역",
    },
])

display(dashboard_design_plan)

,dashboard_page,key_question,main_dataset,recommended_charts,filters
0,1. 전국 문화시설 현황,"시설은 어디에 많고, 관람객 수는 어느 지역에서 높은가?",tableau_facility_main.csv / tableau_region_sum...,"지도, 지역별 막대그래프, KPI 카드","시설 유형, 권역, 시도, 관람객 수 구간"
1,2. 시설 유형별 비교,박물관과 미술관은 운영·규모·관람객 수에서 어떤 차이가 있는가?,tableau_type_summary.csv / tableau_facility_ma...,"박스플롯, 막대그래프, 요약 테이블","시설 유형, 지역, 권역"
2,3. 활성화 요인 분석,관람객 수와 관련된 주요 운영·시설 요인은 무엇인가?,tableau_factor_summary.csv / tableau_facility_...,"변수 중요도 막대그래프, 상관계수 막대그래프, 산점도","요인 그룹, 시설 유형, 지역"
3,4. 서비스 제공 현황,모바일/음성 제공 시설은 어떤 특성을 보이는가?,tableau_facility_main.csv / tableau_region_sum...,"서비스 제공률 막대그래프, 관람객 수 비교 그래프","서비스 제공 여부, 시설 유형, 지역"
4,5. 조건 대비 성과 탐색,비슷한 조건 대비 예상보다 관람객 수가 높은/낮은 시설은 어디인가?,tableau_prediction_performance.csv,"실제 vs 예측 산점도, 오차 막대그래프, 시설 리스트","예측 성과 그룹, 시설 유형, 지역"


## 13. Tableau용 CSV 저장

In [13]:
# 저장 전 컬럼명 확인용 딕셔너리
output_tables = {
    "tableau_facility_main.csv": tableau_facility_main,
    "tableau_region_summary.csv": region_summary,
    "tableau_type_summary.csv": type_summary,
    "tableau_factor_summary.csv": factor_summary,
    "tableau_model_performance.csv": performance,
    "tableau_prediction_performance.csv": tableau_prediction_performance,
    "dashboard_design_plan.csv": dashboard_design_plan,
}

for filename, data in output_tables.items():
    save_path = OUTPUT_DIR / filename
    data.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"saved: {save_path} | shape={data.shape}")

saved: c:\Users\hi\cultural_contest\outputs\tableau\tableau_facility_main.csv | shape=(1191, 50)
saved: c:\Users\hi\cultural_contest\outputs\tableau\tableau_region_summary.csv | shape=(17, 21)
saved: c:\Users\hi\cultural_contest\outputs\tableau\tableau_type_summary.csv | shape=(2, 18)
saved: c:\Users\hi\cultural_contest\outputs\tableau\tableau_factor_summary.csv | shape=(19, 12)
saved: c:\Users\hi\cultural_contest\outputs\tableau\tableau_model_performance.csv | shape=(21, 12)
saved: c:\Users\hi\cultural_contest\outputs\tableau\tableau_prediction_performance.csv | shape=(228, 14)
saved: c:\Users\hi\cultural_contest\outputs\tableau\dashboard_design_plan.csv | shape=(5, 5)


## 14. 저장 결과 확인

In [14]:
saved_files = sorted(OUTPUT_DIR.glob("*.csv"))
for path in saved_files:
    print(path.name)

dashboard_design_plan.csv
tableau_facility_main.csv
tableau_factor_summary.csv
tableau_model_performance.csv
tableau_prediction_performance.csv
tableau_region_summary.csv
tableau_type_summary.csv


## 15. 최종 정리

본 노트북에서는 EDA, 통계분석, 머신러닝 결과를 Tableau 대시보드 제작에 적합한 형태로 정리하였다.

주요 처리 내용은 다음과 같다.

1. 지역명 표준화
   - `강원 → 강원특별자치도`
   - `전북 → 전북특별자치도`
   - `제주 → 제주특별자치도`

2. 시설 단위 메인 데이터셋 생성
   - 시설 유형, 지역, 위치, 운영연수, 규모, 프로그램, 관람객 수, 서비스 제공 여부를 포함하였다.
   - 머신러닝 예측 결과가 있는 시설에는 예측값, 실제값, 오차값을 결합하였다.

3. Tableau 요약 데이터셋 생성
   - 지역 요약
   - 시설 유형 요약
   - 활성화 요인 요약
   - 모델 성능 비교
   - 조건 대비 성과 시설 목록

4. 대시보드 설계 방향 정리
   - 전국 문화시설 현황
   - 시설 유형별 비교
   - 활성화 요인 분석
   - 서비스 제공 현황
   - 조건 대비 성과 탐색

다음 단계에서는 저장된 `outputs/tableau` 폴더의 CSV 파일을 Tableau에 연결하여 대시보드를 제작한다.